In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
from datetime import datetime, timedelta
from time import sleep
import yfinance as yf
import praw
import re
import statsmodels.api as sm
from scipy.stats import ttest_1samp, ttest_ind, pearsonr, spearmanr
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import plotly.express as px
from IPython.display import HTML
from ipywidgets import Tab, Output
from IPython.display import display

In [2]:
releases_df = pd.read_csv('../data/top_releases.csv')
stock_data = pd.read_csv('../data/stock_data.csv')
returns_df = pd.read_csv('../data/abnormal_returns.csv')
engagement_df = pd.read_csv('../data/reddit_engagement.csv')
minimal_posts_df = pd.read_csv('../data/reddit_engagement_minimal.csv')

all_genres = list(releases_df.columns[9:-2])

clean_stock_data = stock_data.dropna()

# Merge returns_df and engagement_df on the 'title' and 'release_date' columns
merged_data = pd.merge(returns_df, engagement_df, on=['name', 'release_date'], how='inner')
# Select only the relevant columns
merged_data = merged_data.drop(columns=['id', 'first_air_date', 'provider_id'])

genre_counts = merged_data[all_genres].sum().sort_values(ascending=False)

abnormal_returns = returns_df['abnormal_return'].dropna()

merged_data['return_positive'] = (merged_data['abnormal_return'] > 0).astype(int)

# Introduction
Streaming platforms have revolutionized how audiences consume content—and Wall Street is paying attention. This study uses an event‐study framework to quantify abnormal stock returns around new TV and movie releases on Netflix and Disney+. We also investigate whether popularity metrics (e.g. TMDB popularity, vote average) or Reddit engagement & sentiment can help explain any price moves.

# Data & Methodology  
1. **Content metadata** from TMDB (release dates, popularity, vote averages, genres).  
2. **Stock prices** for Netflix (NFLX) and Disney (DIS), from 1999 through early 2025, computing daily returns and estimating a market model.  
3. **Event windows** defined around each release:
   $\text{abnormal\_return}_t = \text{actual\_return}_t - (\alpha + \beta \cdot \text{market\_return}_t).$  
4. **Reddit discussion data** via PRAW: number of posts/comments, engagement score, and sentiment analysis per release.


*The full analysis details can be accessed in this [report](https://drive.google.com/file/d/1vTIoDPECp8LruLVELqSwoWXwHAjUA0DX/view?usp=sharing)*

# Key Findings  

### Provider-specific and content-type effects  
- Netflix titles yield an average abnormal return of **+0.43%**, whereas Disney content shows a slight average of **–0.07%** around release day.  
- Netflix releases exhibit a wider spread (outliers up to ±30%), while Disney’s moves are more muted.
- Movies may see larger swings due to blockbuster expectations and one-time release events.  
- TV Shows often build momentum over episodes, potentially leading to more moderated price reactions.  



In [4]:
fig1 = px.box(
    merged_data,
    x='provider',
    y='abnormal_return',
    title='Distribution of Abnormal Returns by Provider',
    labels={'provider':'Provider', 'abnormal_return':'Abnormal Return (%)'}
)

fig1_2 = px.box(
    merged_data,
    x='content-type',
    y='abnormal_return',
    title='Distribution of Abnormal Returns by Content Type',
    labels={'content-type':'Content Type', 'abnormal_return':'Abnormal Return (%)'}
)

out1 = Output()
out2 = Output()

with out1:
    fig1.show()

with out2:
    fig1_2.show()

tab = Tab(children=[out1, out2])
tab.set_title(0, 'By Provider')
tab.set_title(1, 'By Content Type')
display(tab)


### Social media engagement  
There is **no strong linear relationship** between Reddit engagement and abnormal returns (corr ≈ 0.04). High buzz doesn’t necessarily translate to immediate stock gains.


In [5]:
fig2 = px.scatter(
    merged_data,
    x='engagement_score',
    y='abnormal_return',
    title='Engagement Score vs Abnormal Return',
    labels={'engagement_score':'Engagement Score','abnormal_return':'Abnormal Return (%)'}
)
fig2.show()

### Sentiment analysis  
Average sentiment around a release is not meaningfully correlated with abnormal return (corr ≈ –0.02). Sentiment swings appear orthogonal to short-term price moves.


In [6]:
fig3 = px.scatter(
    merged_data,
    x='avg_sentiment',
    y='abnormal_return',
    title='Average Sentiment vs Abnormal Return',
    labels={'avg_sentiment':'Average Sentiment','abnormal_return':'Abnormal Return (%)'}
)
fig3.show()

### Lagged effects
We tested whether Reddit buzz or sentiment one to three days before release predicts abnormal returns on release day. None of the lagged features showed significant predictive power (all p-values > 0.1), suggesting **no reliable lead–lag relationship** in the short window.



In [8]:
# Function to get lagged abnormal returns
# The function calculates the abnormal returns for a specified number of days after the release date
def get_lagged_abnormal_returns(releases, stock_data, lag_days):
    lagged_abnormal_returns = []

    for _, release in releases.iterrows():
        ticker = release['ticker']
        release_date = pd.to_datetime(release['release_date'])
        lag_date = release_date + timedelta(days=lag_days)

        start_date = release_date
        end_date = lag_date
        # Get stock data for this window
        stock_window = stock_data[
            (stock_data['Ticker'] == ticker) & 
            (stock_data['Date'] >= start_date) & 
            (stock_data['Date'] <= end_date)
        ]
        
        # Get market data (S&P 500) for the same window
        market_window = stock_data[
            (stock_data['Ticker'] == 'SPY') & 
            (stock_data['Date'] >= start_date) & 
            (stock_data['Date'] <= end_date)
        ]

        if len(stock_window) > 0 and len(market_window) > 0:
            # Calculate cumulative returns
            if len(stock_window) > 1:
                # Use first and last prices
                first_price = stock_window.iloc[0]['Adj Close']
                last_price = stock_window.iloc[-1]['Adj Close']
                stock_return = (last_price - first_price) / first_price * 100
            else:
                stock_return = stock_window.iloc[0]['Daily_Return']
            
            if len(market_window) > 1:
                first_market = market_window.iloc[0]['Adj Close']
                last_market = market_window.iloc[-1]['Adj Close']
                market_return = (last_market - first_market) / first_market * 100
            else:
                market_return = market_window.iloc[0]['Daily_Return']
            
            abnormal_return = stock_return - market_return
            lagged_abnormal_returns.append(abnormal_return)
        else:
            lagged_abnormal_returns.append(np.nan)

    return lagged_abnormal_returns


stock_data['Date'] = pd.to_datetime(stock_data['Date'])
merged_data['abnormal_return_day1'] = get_lagged_abnormal_returns(merged_data, stock_data, lag_days=1)
merged_data['abnormal_return_day3'] = get_lagged_abnormal_returns(merged_data, stock_data, lag_days=3)
merged_data['abnormal_return_day7'] = get_lagged_abnormal_returns(merged_data, stock_data, lag_days=7)


abnormal_lag_days = ['abnormal_return_day1', 'abnormal_return_day3', 'abnormal_return_day7']
abnormal_lag_days_int = [1, 3, 7]

engagement_metrics = ['popularity', 'vote_average', 'vote_count',
                  'avg_sentiment', 'avg_weighted_sentiment', 'engagement_score']

columns_of_interest = engagement_metrics + abnormal_lag_days + ['abnormal_return']

corr_data = merged_data[columns_of_interest]
corr_matrix = corr_data.corr(method='pearson')


In [9]:
fig4 = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu",
    origin="lower",
    title="Correlation Matrix: Reddit Engagement vs Abnormal Return"
)
fig4.update_layout(
    xaxis_title="Features",
    yaxis_title="Features",
    width=700,
    height=600
)
fig4.show()

The correlation matrix supports the conclusion that there is no meaningful lagged relationship between Reddit engagement or sentiment metrics and streaming providers' abnormal stock returns. Across all lag periods, correlations between engagement variables and abnormal returns remain close to zero. \
While there is a visible correlation between the movie's popularity with its sentiment, it is not useful to our main research question purpose.

## Discussion & Implications  
- The modest average abnormal return for Netflix suggests content drops do move the stock, but only slightly and inconsistently.  
- The absence of strong correlations with popularity, social engagement, or sentiment implies **traditional event-study signals** may not capture investor attention in this space.  
- Provider-level nuances (marketing strategies, subscriber narratives) likely overshadow content-specific buzz.  
- **Limitations & next steps**: 
    - Extend windows to capture post-release trends
    - Segment by genre or region
    - Distinguish TV Show vs. Movie
    - Explore lead-lag effects.


## Conclusion  
While streaming releases generate excitement, their **direct impact on immediate stock returns** is modest and unpredictable. Investors and analysts should consider broader subscriber and financial metrics alongside event-driven buzz when evaluating streaming stocks.